# Heat: are cooler streets calmer for cyclists?
### Enriching GPS trajectories with Earth Observation, OpenStreetMap and Street View Imagery
*Summer School: Earth Observation and GeoAI for Climate-Resilient Urban Systems*

---


## Your research question
> **Do stress hotspots coincide with hot surfaces (land surface temperature)? Does vegetation cool the streets we ride on? Can we route cyclists along cooler streets?**

You will compute greenness with Sentinel-2, surface temperature with the Landsat thermal band, test whether greener cells are cooler, and finally compute a **cool route** with Dijkstra's algorithm.


**How to use:** run cells top to bottom with **Shift + Enter**. In the **✏️ Your turn** tasks, fill in the blanks `____` and run the **✅ check** cell below. Boxes marked **📖 Deep dive** or **🔍 What the code does** are optional.

### 🐍 Python basics
| you see | it means |
|---|---|
| `____` | a blank for you to fill in (four underscores) |
| `df.head()` | first 5 rows of the table `df` |
| `df.column` or `df["column"]` | one column of a table |
| `df[df.speed_kmh < 45]` | keep only the rows where the condition is true |
| `df.groupby("a").b.mean()` | mean of column `b` for every value of column `a` |
| `(cond1) & (cond2)` | both conditions must be true (put each in brackets!) |
| `f"{x:.2f}"` | text with the value of `x` rounded to 2 decimals |


## 0 · Setup  
Run the next three cells. They install a few packages, set the options and load the data and helper functions.

<details><summary>🔍 Which packages are installed?</summary>

| package | purpose |
|---|---|
| `osmnx` | download OpenStreetMap data |
| `pystac-client`, `planetary-computer`, `odc-stac` | search and load satellite images from Microsoft's Planetary Computer (STAC catalogue) |
| `mapclassify` | quantile classes for map colours |
| `h3` | Uber's hexagonal grid (optional alternative to square cells) |
| `ultralytics` | YOLO26 object detection |
</details>

In [ ]:
# Install the few packages Colab does not ship with (takes ~1 min)
!pip install -q osmnx pystac-client planetary-computer odc-stac mapclassify h3 ultralytics

<details><summary>⚙️ Configuration options (defaults work — change only if you want to)</summary>

- `LIVE`: `True` = try fresh downloads first; `False` = use only the pre-downloaded data. Any failed download automatically falls back to the pre-downloaded data.
- `MAPILLARY_TOKEN` *(optional)*: needed only to download street view images live ([get a free client token](https://www.mapillary.com/dashboard/developers), starts with `MLY|`).
- `CARTO_API_KEY` *(optional)*: free key for the light-grey CARTO background map; without it, maps use OpenStreetMap tiles.
- `GRID_TYPE`: `"square"` (100 m cells) or `"h3"` (hexagons, `H3_RES = 10` ≈ 13 000 m²). Try both after the workshop to see the MAUP in action.
- `DATA_URL`: direct download link of the workshop data.
</details>

In [ ]:
# ⚙️ Configuration
LIVE = True            # try live downloads first; set False to use only the pre-downloaded fallback
MAPILLARY_TOKEN = ""  # optional: your client token from https://www.mapillary.com/dashboard/developers (starts with "MLY|")
CARTO_API_KEY = ""       # optional: free key from https://carto.com/basemaps/apikey/ for the light-grey CARTO background map

GRID_TYPE = "square"   # spatial unit: "square" (100 m x 100 m cells) or "h3" (hexagons)
CELL = 100             # side length of square cells in metres          (used if GRID_TYPE == "square")
H3_RES = 10            # H3 resolution: 9 ≈ 105 000 m², 10 ≈ 13 000 m², 11 ≈ 1 900 m²  (used if GRID_TYPE == "h3")

# Where the workshop data (GPS subset + fallbacks) is hosted: a DIRECT download link to the zip.
# Alternatively upload workshop_data.zip via the 📁 file panel on the left.
DATA_URL = "https://raw.githubusercontent.com/mms1068467/IEEE_GRSS_multimodal_mobility_data/main/workshop_data.zip"

The next cell downloads the data, defines the 4 × 4 km study area around Osnabrück's centre and a few helper functions. You do not need to read it.

<details><summary>🔍 What the code does</summary>

- **CRS:** GPS gives degrees (`EPSG:4326`), which are not a distance unit. All data is projected to **ETRS89 / UTM 32N (`EPSG:25832`)**, in metres.
- **`get_data(name, fetch, kind)`** tries the live download `fetch()`; if it fails, it loads `workshop_data/<name>` instead.
- **`cell_id(x, y)`** gives the grid cell of a point (square: `"column_row"`, e.g. `"4348_57922"`; H3: a hexagon ID).
- **`cells_to_gdf(df)`** turns a table indexed by cell ID into polygons for mapping.
- **`raster_to_cells(da, name)`** averages all raster pixels inside each cell (**zonal mean**).
- **`show_map(gdf, column)`** draws an interactive map with a switchable satellite background.
</details>

In [ ]:
import os, zipfile, urllib.request, warnings, json
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import folium, xyzservices.providers as xyz
from shapely.geometry import box
warnings.filterwarnings("ignore")

# ======================= 1. get the workshop data =======================
DATA_DIR = Path("workshop_data")

def _direct_link(url):
    """Turn common 'share page' links into direct file downloads."""
    if "github.com" in url and "/blob/" in url: url = url.replace("/blob/", "/raw/")
    if "dropbox.com" in url: url = url.replace("dl=0", "dl=1")
    return url

def _extract(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        # works whether the zip contains the folder 'workshop_data/' or just the files
        z.extractall("." if any(n.startswith("workshop_data/") for n in names) else DATA_DIR)

def get_workshop_data():
    if (DATA_DIR / "gps_osnabrueck.parquet").exists():
        return
    DATA_DIR.mkdir(exist_ok=True)
    if Path("gps_osnabrueck.parquet").exists():                 # parquet uploaded on its own
        Path("gps_osnabrueck.parquet").rename(DATA_DIR / "gps_osnabrueck.parquet"); return
    for z in sorted(Path(".").glob("workshop_data*.zip")):      # zip uploaded via Colab's file panel
        if zipfile.is_zipfile(z): _extract(z); return
        z.unlink()                                               # remove broken/partial downloads
    if "<your-account>" in DATA_URL:
        raise RuntimeError("Set DATA_URL in the configuration cell, or upload workshop_data.zip "
                           "via the 📁 file panel on the left, then re-run this cell.")
    url = _direct_link(DATA_URL)
    print("Downloading workshop data …")
    if "drive.google.com" in url:                               # Google Drive share links need gdown
        import gdown; gdown.download(url, "workshop_data.zip", quiet=False, fuzzy=True)
    else:
        urllib.request.urlretrieve(url, "workshop_data.zip")
    if not zipfile.is_zipfile("workshop_data.zip"):
        head = Path("workshop_data.zip").read_bytes()[:150]
        Path("workshop_data.zip").unlink()
        raise RuntimeError("DATA_URL did not return a zip file (it probably points to a web page, "
                           f"not the file itself). First bytes received: {head!r}")
    _extract("workshop_data.zip")

get_workshop_data()
print("Data folder:", sorted(p.name for p in DATA_DIR.iterdir()))

# ======================= 2. study area: 4 x 4 km around Osnabrück's city centre =======================
CRS = "EPSG:25832"                      # ETRS89 / UTM zone 32N – metres, official CRS for western Germany
X0, Y0, HALF = 434750, 5792250, 2000    # centre (easting, northing) and half side length in metres
AREA = gpd.GeoDataFrame(geometry=[box(X0-HALF, Y0-HALF, X0+HALF, Y0+HALF)], crs=CRS)
BBOX_WGS84 = tuple(AREA.to_crs(4326).total_bounds)      # (west, south, east, north) in degrees

# ======================= 3. helper functions =======================
# ---------- live download with automatic fallback ----------
def _save(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(obj, gpd.GeoDataFrame): obj.to_parquet(path)
    elif isinstance(obj, pd.DataFrame):   obj.to_parquet(path)
    else:  # xarray.Dataset -> table of pixels
        obj.to_dataframe().reset_index()[["y", "x", *obj.data_vars]].to_parquet(path)

def _load(path, kind):
    if kind == "gdf":    return gpd.read_parquet(path)
    if kind == "df":     return pd.read_parquet(path)
    if kind == "raster": return pd.read_parquet(path).set_index(["y", "x"]).to_xarray()

def get_data(name, fetch, kind):
    """Try the live download; on any problem fall back to workshop_data/<name>."""
    path = DATA_DIR / name
    if LIVE:
        try:
            obj = fetch()
            _save(obj, path)
            print(f"✅ {name}: live download OK")
            return obj
        except Exception as e:
            print(f"⚠️ {name}: live download failed ({type(e).__name__}: {str(e)[:120]}) → fallback")
    obj = _load(path, kind)
    print(f"📦 {name}: loaded fallback")
    return obj

# ---------- grid helpers (square cells or H3 hexagons, see GRID_TYPE) ----------
assert GRID_TYPE in ("square", "h3"), 'GRID_TYPE must be "square" or "h3"'
if GRID_TYPE == "h3":
    import h3
    from pyproj import Transformer
    from shapely.geometry import Polygon
    _to_wgs84 = Transformer.from_crs(CRS, 4326, always_xy=True)

def cell_id(x, y):
    """ID of the grid cell that contains (x, y) given in EPSG:25832."""
    if GRID_TYPE == "square":   # 'column_row', e.g. '4348_57922'
        return (np.floor(np.asarray(x) / CELL).astype(int).astype(str).astype(object) + "_" +
                np.floor(np.asarray(y) / CELL).astype(int).astype(str).astype(object))
    lon, lat = _to_wgs84.transform(np.asarray(x), np.asarray(y))      # H3 works on lat/lon
    return np.array([h3.latlng_to_cell(la, lo, H3_RES) for la, lo in zip(lat, lon)], dtype=object)

def cells_to_gdf(df):
    """Turn a table indexed by cell_id into polygons (squares or hexagons)."""
    if GRID_TYPE == "square":
        ij = df.index.to_series().str.split("_", expand=True).astype(int)
        geom = [box(i*CELL, j*CELL, (i+1)*CELL, (j+1)*CELL) for i, j in zip(ij[0], ij[1])]
        return gpd.GeoDataFrame(df, geometry=geom, crs=CRS)
    geom = [Polygon([(lo, la) for la, lo in h3.cell_to_boundary(c)]) for c in df.index]
    return gpd.GeoDataFrame(df, geometry=geom, crs=4326).to_crs(CRS)

CELL_LABEL = f"{CELL} m square" if GRID_TYPE == "square" else f"H3 res {H3_RES} hexagon"
print("Spatial unit:", CELL_LABEL)

def raster_to_cells(da, name):
    """Average all pixels of a raster (xarray.DataArray in EPSG:25832) per grid cell (zonal mean)."""
    t = da.to_dataframe(name).reset_index()[["x", "y", name]].dropna()
    t["cell_id"] = cell_id(t.x, t.y)              # pixel centre -> cell it falls into
    return t.groupby("cell_id")[name].mean()

# ---------- interactive maps ----------
# CARTO tiles need a (free) API key; without one we fall back to the standard OpenStreetMap tiles
BASEMAP = xyz.CartoDB.Positron(apikey=CARTO_API_KEY) if CARTO_API_KEY else xyz.OpenStreetMap.Mapnik

def show_map(gdf, column, **kwargs):
    """Interactive choropleth map with a switchable satellite background."""
    m = gdf.explore(column=column, tiles=BASEMAP, **kwargs)
    folium.TileLayer(xyz.Esri.WorldImagery, name="Satellite (Esri)", overlay=False).add_to(m)
    folium.LayerControl().add_to(m)
    return m

print("Study area bbox (WGS84):", np.round(BBOX_WGS84, 4))

In [ ]:
def check(condition, ok="✅ Well done!", hint="❌ Not quite yet – look at the hints above the task."):
    """Friendly feedback for the ✏️ Your turn tasks."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        passed, hint = False, f"❌ Not finished yet ({type(e).__name__}) – fill in all blanks `____` and run the task cell first."
    print(ok if passed else hint)

def xr_corr(a, b):
    """Correlation of two rasters (ignoring missing pixels)."""
    m = np.isfinite(a.values) & np.isfinite(b.values)
    return np.corrcoef(a.values[m], b.values[m])[0, 1]


## 1 · Load & explore the stress trajectories 

- A **trajectory** is a time-ordered sequence of points $(x_i, y_i, t_i)$ — here **one point per second (1 Hz)**, with speed and heading. GPS error is about **3–5 m** under open sky and up to tens of metres between tall buildings.

| column | meaning |
|---|---|
| `rider_id`, `ride_id` | pseudonymous IDs of person and trip |
| `time_utc`, `lat`, `lon` | timestamp (UTC) and position (WGS84) |
| `speed_kmh`, `bearing` | GPS speed and heading |
| `mos_score` | stress score from EDA (0–2) |
| `detected_mos` | 1 = Moment of Stress |

The data is pseudonymised, and the first/last 200 m of each ride were removed to protect home locations.


In [ ]:
gps = pd.read_parquet(DATA_DIR / "gps_osnabrueck.parquet")                    # 1. read the table
gps = gpd.GeoDataFrame(gps, geometry=gpd.points_from_xy(gps.lon, gps.lat),    # 2. make points (degrees)
                       crs=4326).to_crs(CRS)                                  # 3. reproject to metres
gps["x"], gps["y"] = gps.geometry.x, gps.geometry.y                           # metric coordinates as columns

print(f"{len(gps):,} GPS points · {gps.ride_id.nunique()} rides · {gps.rider_id.nunique()} riders")
print("period:", gps.time_utc.min().date(), "→", gps.time_utc.max().date())
gps.head()

**Quick look at the data.** What do you notice about speed (peak at 0?), the stress score and the months of data collection?


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
gps.speed_kmh.clip(upper=60).hist(bins=60, ax=ax[0]); ax[0].set_title("speed (km/h)")
gps.mos_score.value_counts().sort_index().plot.bar(ax=ax[1]); ax[1].set_title("mos_score (log scale)"); ax[1].set_yscale("log")
gps.time_utc.dt.to_period("M").value_counts().sort_index().plot.bar(ax=ax[2]); ax[2].set_title("points per month")
plt.tight_layout()

**Cleaning.** We drop implausible speeds (> 45 km/h: GPS errors or a ride in a car or bus) and flag points below 2 km/h as `stopped` (waiting at lights, pushing the bike) — we keep them, because waiting may itself be stressful.

In [ ]:
n0 = len(gps)
gps = gps[gps.speed_kmh < 45].copy()     # drop implausible speeds
gps["stopped"] = gps.speed_kmh < 2      # True/False column: is the cyclist (almost) standing still?
print(f"removed {n0 - len(gps):,} implausible points; {gps.stopped.mean():.0%} of remaining points are stops")

### ✏️ Your turn 1
Is stress more frequent when cyclists are **stopped** or **moving**?

Every row is one second of riding. `detected_mos` is 1 for a stressed second and 0 otherwise, so the **mean** of `detected_mos` is the *share* of stressed seconds. Multiply that share by 3600 (seconds per hour) to get **stress events per hour**.

Fill in the blanks: group by the column that says whether a cyclist is stopped, take the mean, multiply by 3600.

<details><summary>🆘 Hint</summary>

The column created in the cleaning step is called `stopped`. The function for the average is `.mean()`.
</details>

In [ ]:
stress_per_hour = gps.groupby("____").detected_mos.____() * ____
stress_per_hour

In [ ]:
check(lambda: set(stress_per_hour.index) == {True, False} and 0 < stress_per_hour.max() < 3600,
      ok="✅ Well done! Which group has more stress events per hour?")

> 💬 **Point for discussion:** Why could stopping go together with stress? Think of traffic lights, conflicts with cars at junctions, etc. 

## 2 · From points to places: a grid of cells 

Our points describe **people**; we want to learn about **places**. We therefore aggregate all points inside the same **100 m × 100 m cell**. This smooths out GPS error and the physiological delay, turns rare events into stable rates, and gives us a common unit for attaching satellite, OSM and street view data.

Our outcome is the **stress rate** — Moments of Stress per hour of riding in a cell.

<details><summary>🔍 What the code does</summary>

`cell_id` assigns each point to a cell; `groupby("cell_id").agg(...)` computes one row per cell: seconds of riding (`n_points`), distinct rides and riders, mean stress score, stress rate, median speed and share of time stopped.
</details>

In [ ]:
gps["cell_id"] = cell_id(gps.x, gps.y)            # assign each point to its grid cell (square or H3)

cells = gps.groupby("cell_id").agg(
    n_points    = ("ride_id", "size"),             # seconds of riding in the cell
    n_rides     = ("ride_id", "nunique"),          # distinct trips
    n_riders    = ("rider_id", "nunique"),         # distinct people
    mos_mean    = ("mos_score", "mean"),           # mean stress score
    stress_rate = ("detected_mos", lambda s: s.mean() * 3600),   # stress events per hour of riding (see above)
    speed_med   = ("speed_kmh", "median"),
    stop_share  = ("stopped", "mean"),             # share of time standing still
)
print(len(cells), f"cells ({CELL_LABEL}) with data")
cells.describe().round(2)

### ✏️ Your turn 2
A cell visited by one rider on one day tells us more about that **person** than about the **place**.
Keep only cells with **at least 10 rides and at least 3 different riders**.

<details><summary>🆘 Hint</summary>

"At least" means `>=`. The columns are `n_rides` and `n_riders`; the thresholds are stored in `MIN_RIDES` and `MIN_RIDERS`. Both conditions must be true: `(condition 1) & (condition 2)`.
</details>

In [ ]:
MIN_RIDES, MIN_RIDERS = 10, 3
cells_original = cells.copy()                      # keep a copy for comparison
cells = cells[(cells.n_rides >= ____) & (cells.____ >= MIN_RIDERS)]
print(len(cells), "reliable cells out of", len(cells_original), f"({len(cells)/len(cells_original):.0%} kept)")

In [ ]:
check(lambda: len(cells) < len(cells_original) and (cells.n_rides >= 10).all() and (cells.n_riders >= 3).all())

**Stress map.** Colours use **quantiles** (each of the 5 classes holds 20 % of the cells) — check the legend for the real values. Switch to the satellite background with the layer button (top right).

In [ ]:
cells_gdf = cells_to_gdf(cells)
show_map(cells_gdf, "stress_rate", cmap="Reds", scheme="quantiles", k=5,
         legend_kwds={"caption": "stress events / hour"})

> 💬 **Point for discussion:** Where do stress hotspots occur — junctions, main roads, the old town? Would the picture change with 50 m or 250 m cells (MAUP)?

## 3 · Sentinel-2: greenness 

 We use the **Sentinel-2** Level-2A **surface reflectance** product, mask clouds with its scene classification layer and take the per-pixel **median** of the clearest summer-2023 scenes.

Vegetation absorbs **red** light but strongly reflects **near-infrared (NIR)**; built-up surfaces reflect more **short-wave infrared (SWIR)** than NIR:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}} \qquad\qquad \text{NDBI} = \frac{\text{SWIR} - \text{NIR}}{\text{SWIR} + \text{NIR}}$$

NDVI: < 0 water · 0–0.2 sealed surfaces · 0.2–0.5 sparse vegetation · > 0.6 dense vegetation.

<details><summary>📖 Deep dive: bands</summary>

| band | name | wavelength | resolution |
|---|---|---|---|
| B04 | red | 665 nm | 10 m |
| B08 | NIR | 842 nm | 10 m |
| B11 | SWIR | 1610 nm | 20 m |

</details>

In [ ]:
def fetch_sentinel2():
    import pystac_client, planetary_computer, odc.stac, xarray as xr
    # 1. open the STAC catalogue
    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                        modifier=planetary_computer.sign_inplace)
    # 2. search scenes: our area, summer 2023, < 20 % clouds
    items = catalog.search(collections=["sentinel-2-l2a"], bbox=BBOX_WGS84,
                           datetime="2023-06-01/2023-08-31",
                           query={"eo:cloud_cover": {"lt": 20}}).item_collection()
    items = sorted(items, key=lambda i: i.properties["eo:cloud_cover"])[:8]     # 8 least cloudy scenes
    print(f"using {len(items)} scenes:", [i.datetime.date().isoformat() for i in items])
    # 3. load only our study area, reprojected to EPSG:25832 at 10 m
    ds = odc.stac.load(items, bands=["B04", "B08", "B11", "SCL"], bbox=BBOX_WGS84,
                       crs=CRS, resolution=10, groupby="solar_day")
    # 4. clear-sky mask: vegetation, not vegetated, water, unclassified
    clear = ds.SCL.isin([4, 5, 6, 7])
    # 5. digital numbers -> reflectance, mask clouds, median over time
    out = {}
    for band, name in [("B04", "red"), ("B08", "nir"), ("B11", "swir")]:
        refl = (ds[band].astype("float32") - 1000) / 10000   # L2A baseline ≥ 04.00 has a -1000 offset
        out[name] = refl.where(clear & (ds[band] > 0)).median("time").clip(0, 1)
    return xr.Dataset(out)

s2 = get_data("s2_summer2023.parquet", fetch_sentinel2, "raster")
s2

`s2` is an **`xarray.Dataset`** of labelled 2-D arrays (`red`, `nir`, `swir`). Arithmetic works pixel by pixel, so NDVI is written exactly like the formula:

In [ ]:
s2["ndvi"] = (s2.nir - s2.red) / (s2.nir + s2.red)        # pixel-wise, like the formula above
s2.ndvi.plot(cmap="RdYlGn", vmin=-0.2, vmax=0.9, figsize=(6, 5)); plt.title("NDVI, summer 2023 median");

**From pixels to cells (zonal statistics).** Each 100 m cell contains 10 × 10 Sentinel-2 pixels; `raster_to_cells` averages them.

In [ ]:
cells["ndvi"] = raster_to_cells(s2.ndvi, "ndvi")
cells[["stress_rate", "ndvi"]].describe().round(2)

## 4 · Landsat 8/9: land surface temperature  ⏱ 10 min

Sentinel-2 has no thermal band, so we use **Landsat 8 and 9**. Their thermal sensors measure emitted infrared radiation at **100 m**; the USGS Level-2 product is already corrected for the atmosphere and surface emissivity. Digital numbers are converted to temperature with
$\text{LST}\,[^\circ\text{C}] = \text{DN} \times 0.00341802 + 149.0 - 273.15$; clouds and shadows are masked with the `qa_pixel` band.

> 🌡️ LST is the temperature of the surface itself. Higher values = sealed, sun-exposed surfaces; lower values = vegetation, shade or water.



In [ ]:
def fetch_landsat():
    import pystac_client, planetary_computer, odc.stac, xarray as xr
    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                        modifier=planetary_computer.sign_inplace)
    items = catalog.search(collections=["landsat-c2-l2"], bbox=BBOX_WGS84,
                           datetime="2023-06-01/2023-08-31",
                           query={"eo:cloud_cover": {"lt": 30},
                                  "platform": {"in": ["landsat-8", "landsat-9"]}}).item_collection()
    print(f"using {len(items)} scenes:", [i.datetime.date().isoformat() for i in items])
    ds = odc.stac.load(items, bands=["lwir11", "qa_pixel"], bbox=BBOX_WGS84,
                       crs=CRS, resolution=30, groupby="solar_day")
    cloudy = (ds.qa_pixel.astype("uint16") & 0b11010) > 0     # bits 1 dilated cloud, 3 cloud, 4 shadow
    lst = (ds.lwir11.astype("float32") * 0.00341802 + 149.0 - 273.15)   # DN -> Kelvin -> °C
    lst = lst.where(~cloudy & (ds.lwir11 > 0))                           # mask clouds and no-data
    return xr.Dataset({"lst": lst.median("time")})

ls = get_data("landsat_lst_summer2023.parquet", fetch_landsat, "raster")
ls.lst.plot(cmap="inferno", figsize=(6, 5)); plt.title("Land surface temperature (°C), summer 2023 median");

In [ ]:

cells["lst"] = raster_to_cells(ls.lst, "lst")      # ~11 Landsat pixels (30 m) per 100 m square
cells["lst"].describe().round(1)

**Compare the layers.** Switch between LST and NDVI with the layer button (top right). Where are green cells cool — and where not?

In [ ]:
# Show the interactive map with togglable layers for LST, NDVI, and NDBI
import folium

gdf = cells_to_gdf(cells)
TIPS = [c for c in ["lst", "ndvi", "ndbi", "stress_rate"] if c in gdf]   # works even without NDBI

# Initialize the interactive map focused on our data
m = gdf.explore(column="lst", cmap="inferno", name="LST (°C)", fill_opacity=0.6, line_opacity=0.2,
                tiles=BASEMAP, tooltip=TIPS)

# Add the NDVI layer
gdf.explore(m=m, column="ndvi", cmap="RdYlGn", name="NDVI (Greenness)", fill_opacity=0.6, line_opacity=0.2,
            tooltip=TIPS)

# Add the NDBI layer (if Your turn 3 is done)
if "ndbi" in gdf:
    gdf.explore(m=m, column="ndbi", cmap="RdGy_r", name="NDBI (Built-up)", fill_opacity=0.6, line_opacity=0.2,
                tooltip=TIPS)

# Add standard satellite background layer option
folium.TileLayer(xyz.Esri.WorldImagery, name="Satellite (Esri)", overlay=False).add_to(m)

# Add layer control so the user can toggle between them
folium.LayerControl().add_to(m)
m

### ✏️ Your turn 3
Where are the **hottest** places? Select the **10 cells with the highest `lst`** and compare their stress rate with the average of all cells.

<details><summary>🆘 Hint</summary>

`table.nlargest(n, "column")` returns the n rows with the largest values in that column.
</details>

In [ ]:
hottest = cells.____(10, "____")
print(f"stress rate – hottest 10 cells: {hottest.stress_rate.mean():.0f} /h · all cells: {cells.stress_rate.mean():.0f} /h")
hottest[["lst", "ndvi", "stress_rate"]].round(2)

In [ ]:
check(lambda: len(hottest) == 10 and hottest.lst.min() >= cells.lst.sort_values().iloc[-10],
      ok="✅ Well done! Are the hottest cells also the most stressful? Find them on the map above.")

### ✏️ Your turn 4
Does vegetation cool the streets we ride on? Make a **scatter plot** of greenness (`ndvi`) against surface temperature (`lst`) and compute their **Spearman correlation**.

<details><summary>🆘 Hint</summary>

`cells.plot.scatter(x_column, y_column)` puts the first column on the x-axis. The correlation method is given as text: `"spearman"`. Afterwards, try `"pearson"` — why can the two values differ?
</details>

In [ ]:
ax = cells.plot.scatter("____", "____", alpha=.4, figsize=(5, 4))
r = cells[["ndvi", "lst"]].corr(method="____").iloc[0, 1]    # the value off the diagonal = the correlation
ax.set_title(f"Spearman r = {r:.2f}");

In [ ]:
check(lambda: -1 <= r <= 1 and ax.get_xlabel() == "ndvi" and ax.get_ylabel() == "lst",
      ok="✅ Well done! Is r negative (greener cells are cooler) or positive?",
      hint="❌ Not quite – put ndvi on the x-axis, lst on the y-axis, and use \"spearman\".")

> 💬 **Point for discussion:** The imagery is from **summer**, but most rides were in **autumn/winter**. Is summer LST still meaningful — e.g. for *where* heat would be a problem?

## 5 · Putting it together 

Which of our features go together with stress? We use **Spearman's ρ**: it correlates the *ranks* of the values, so it captures any monotonic relation ("the more A, the more B") and is robust to skewed data. ρ = +1 perfectly increasing, 0 none, −1 perfectly decreasing; roughly, |ρ| < 0.1 is negligible and |ρ| > 0.3 moderate.


In [ ]:
features = ['lst', 'ndvi', 'stop_share', 'speed_med']
features = [f for f in features if f in cells]   # skips columns of unfinished tasks
corr = cells[features + ["stress_rate", "mos_mean"]].corr(method="spearman")[["stress_rate", "mos_mean"]].loc[features]
corr.sort_values("stress_rate").plot.barh(figsize=(7, 4), color=["tab:red", "tab:orange"])
plt.axvline(0, color="k", lw=.8); plt.title(f"Spearman correlation with stress (per {CELL_LABEL})"); plt.tight_layout()
corr.round(2)

The enriched grid — hover over a cell to see its values. Change `"lst"` to `"stress_rate"` to compare the patterns.

In [ ]:
show_map(cells_to_gdf(cells), "lst", cmap="inferno",
         tooltip=[c for c in ["stress_rate"] + ['lst', 'ndvi', 'stop_share', 'speed_med'] if c in cells])

## 6 · Cool routing  ⏱ 8 min

Can our results help cyclists choose a better route? A street network is a **graph**: intersections are **nodes**, street segments are **edges** with a **cost**. **Dijkstra's algorithm** finds the path with the lowest total cost between two points. The algorithm never changes — only the **edge cost** does:

| route | edge cost |
|---|---|
| 🔵 Shortest | `length` |
| 🟢 Coolest | `length × (1 + weight × penalty)` |


<details><summary>🔍 What the code does</summary>

1. Download the cycling street network with `osmnx` (or load the saved copy) and project it to metres.
2. Copy `lst`, `ndvi` and `stress_rate` from the grid cells to the street segments; `scale01` rescales values to 0–1.
3. Compute the **penalty** of every segment and the cost `rq_cost`; `add_costs(weight)` rebuilds the routable graph.
4. `compare_routes()` runs `nx.shortest_path` for both costs and reports length, detour and the mean LST, NDVI and stress rate along each route.
</details>

In [ ]:
import osmnx as ox, networkx as nx
from shapely.geometry import Point, LineString

GRAPH_FILE = DATA_DIR / "osm_bike_graph.graphml"

def load_street_graph():
    # live download with fallback to the saved copy (same idea as get_data)
    if LIVE or not GRAPH_FILE.exists():
        try:
            G = ox.graph_from_bbox(bbox=BBOX_WGS84, network_type="bike", simplify=True)
            ox.save_graphml(G, GRAPH_FILE)
            print("✅ street network: live download OK"); return G
        except Exception as e:
            print(f"⚠️ street network: live download failed ({type(e).__name__}) → fallback")
    G = ox.load_graphml(GRAPH_FILE); print("📦 street network: loaded fallback"); return G

G = ox.project_graph(load_street_graph(), to_crs=CRS)
nodes, edges = ox.graph_to_gdfs(G)

# every street segment takes the values of the grid cell its centre point lies in
JOIN_COLS = [c for c in ['lst', 'ndvi', 'stress_rate'] if c in cells]
centres = gpd.GeoDataFrame(geometry=edges.geometry.centroid, crs=CRS)
joined = gpd.sjoin(centres, cells_to_gdf(cells)[JOIN_COLS + ["geometry"]], how="left", predicate="within")
for col in JOIN_COLS:
    edges[col] = joined[col].reindex(edges.index).fillna(cells[col].mean())   # outside reliable cells → mean

def scale01(s):
    """Rescale to 0–1 (5th percentile → 0, 95th percentile → 1)."""
    lo, hi = s.quantile(.05), s.quantile(.95)
    return ((s - lo) / (hi - lo)).clip(0, 1)

edges["penalty"] = scale01(edges["lst"])              # 0 = coolest, 1 = hottest

def add_costs(weight=1.0):
    # cost = length × (1 + weight × penalty): with weight 0 every route is the shortest one
    edges["rq_cost"] = edges["length"] * (1.0 + weight * edges["penalty"])
    return ox.graph_from_gdfs(nodes, edges)

G_weighted = add_costs()
print(f"{len(nodes):,} intersections · {len(edges):,} street segments · mean penalty {edges.penalty.mean():.2f}")

In [ ]:
SUMMARY_COLS = ['lst', 'ndvi', 'stress_rate']

ORIG_XY = (433200, 5791200)    # south-west of the centre (EPSG:25832, metres)
DEST_XY = (436100, 5793000)    # north-east of the centre
ROUTES = {"Shortest": ("length", "blue"), "Coolest": ("rq_cost", "green")}

def edge_used(graph, u, v, weight):
    return min(graph[u][v].values(), key=lambda d: d[weight])        # cheapest of parallel edges

def compare_routes(graph, orig_xy=ORIG_XY, dest_xy=DEST_XY):
    o = ox.nearest_nodes(graph, X=orig_xy[0], Y=orig_xy[1])
    d = ox.nearest_nodes(graph, X=dest_xy[0], Y=dest_xy[1])
    rows, lines = [], []
    for name, (weight, colour) in ROUTES.items():
        path = nx.shortest_path(graph, o, d, weight=weight)           # Dijkstra
        segs = [edge_used(graph, u, v, weight) for u, v in zip(path[:-1], path[1:])]
        length = sum(s["length"] for s in segs)
        row = {"route": name, "length_km": length / 1000}
        for col in SUMMARY_COLS:                                       # length-weighted mean along the route
            row[col] = sum(s["length"] * s[col] for s in segs) / length
        rows.append(row)
        pts = [Point(graph.nodes[n]["x"], graph.nodes[n]["y"]) for n in path]
        lines.append({"Route": name, "Distance": f"{length/1000:.2f} km", "geometry": LineString(pts)})
    table = pd.DataFrame(rows).set_index("route")
    table["detour_%"] = (table.length_km / table.loc["Shortest", "length_km"] - 1) * 100
    m = gpd.GeoDataFrame(lines, crs=CRS).explore(column="Route", cmap=[c for _, c in ROUTES.values()],
                                                 categories=list(ROUTES), tiles=BASEMAP,
                                                 style_kwds=dict(weight=5), tooltip=["Route", "Distance"])
    folium.TileLayer(xyz.Esri.WorldImagery, name="Satellite (Esri)", overlay=False).add_to(m)
    folium.LayerControl().add_to(m)
    display(table.round(2))
    return m

compare_routes(G_weighted)

### ✏️ Your turn 5
With `weight = 1` the coolest route may be identical to (or only slightly different from) the shortest one.
**Increase the weight** (try `3`, then `10`) and compare again. How much detour (%) would *you* accept for a cooler ride?

<details><summary>🆘 Hint</summary>

Write a number instead of `____`, e.g. `3`. Larger weight = hot streets look "longer" to the algorithm.
</details>

In [ ]:
G_weighted = add_costs(weight=____)
compare_routes(G_weighted)

> 💬 **Point for discussion:** Is the coolest route also greener and calmer? LST is one summer-morning snapshot of the *surface* — what would a heat-aware routing app really need (time of day, shade, air temperature)?

---
## ⭐ Bonus: shade potential (buildings & trees) and a shaded route
NDVI and LST do not describe shade at street level. Shade comes from **tree crowns** and **buildings**. We download trees and building footprints from OSM, compute the share of each cell covered by buildings and by tree crowns (15 m radius per mapped tree) and combine them into a **shade potential** (40 % buildings, 60 % trees).

In [ ]:
OSM_TAGS = {
    "highway": ["traffic_signals", "cycleway", "primary", "secondary", "tertiary"],
    "natural": ["tree"],
    "cycleway": True, "cycleway:both": True, "cycleway:left": True, "cycleway:right": True,
}
def fetch_osm():
    import osmnx as ox
    f = ox.features_from_bbox(BBOX_WGS84, tags=OSM_TAGS).reset_index()     # bbox = (west, south, east, north)
    cols = ["element", "id", "highway", "natural", "cycleway", "cycleway:both", "cycleway:left", "cycleway:right", "geometry"]
    return f.reindex(columns=cols).to_crs(CRS)

osm = get_data("osm_features.parquet", fetch_osm, "gdf")

signals = osm[osm.highway == "traffic_signals"]
trees   = osm[osm.natural == "tree"]
lines   = osm[osm.geom_type.isin(["LineString", "MultiLineString"])]
cycle_infra = lines[(lines.highway == "cycleway") |                                   # separate cycle paths …
                    lines[["cycleway", "cycleway:both", "cycleway:left", "cycleway:right"]]
                        .isin(["lane", "track", "separate", "shared_lane"]).any(axis=1)]   # … or lanes on roads
busy_roads = lines[lines.highway.isin(["primary", "secondary", "tertiary"])]
print(f"{len(signals)} traffic signals · {len(trees)} trees · "
      f"{lines.length.sum()/1000:.0f} km in total · "
      f"{cycle_infra.length.sum()/1000:.0f} km cycle infrastructure · {busy_roads.length.sum()/1000:.0f} km busy roads")

In [ ]:
import osmnx as ox
import geopandas as gpd

# 1. building footprints from OSM
print("Loading building footprints from OSM …")
building_tags = {"building": True}
try:
    buildings = ox.features_from_bbox(BBOX_WGS84, tags=building_tags).to_crs(CRS)
    print(f"{len(buildings)} buildings loaded.")
except Exception as e:
    print(f"Could not load buildings: {e}")
    buildings = gpd.GeoDataFrame(geometry=[], crs=CRS)

# Filter to only polygon and multipolygon geometries to avoid mixed geometry type error in gpd.overlay
if not buildings.empty:
    buildings = buildings[buildings.geom_type.isin(["Polygon", "MultiPolygon"])]

# 2. built-up area per cell (shade cast by buildings)
if not buildings.empty:
    # intersect buildings with the grid cells
    b_parts = gpd.overlay(buildings[["geometry"]], cells_gdf.reset_index()[["cell_id", "geometry"]], how="intersection")
    b_parts["area"] = b_parts.geometry.area
    building_area_cell = b_parts.groupby("cell_id")["area"].sum().reindex(cells.index, fill_value=0)
    # share of the cell covered by buildings (0-1)
    cells["building_density"] = building_area_cell / (CELL * CELL)
else:
    cells["building_density"] = 0.0

# 3. tree crowns: assume a 15 m crown radius per mapped tree
if not trees.empty:
    trees_buffered = trees.copy()
    trees_buffered["geometry"] = trees_buffered.geometry.buffer(15)  # 15 m crown radius
    t_parts = gpd.overlay(trees_buffered[["geometry"]], cells_gdf.reset_index()[["cell_id", "geometry"]], how="intersection")
    t_parts["area"] = t_parts.geometry.area
    tree_area_cell = t_parts.groupby("cell_id")["area"].sum().reindex(cells.index, fill_value=0)
    cells["tree_shadow_density"] = (tree_area_cell / (CELL * CELL)).clip(0, 1)
else:
    cells["tree_shadow_density"] = 0.0

# 4. combined shade index
# weighted combination of building density and tree-crown shade
cells["shade_potential"] = (cells["building_density"] * 0.4 + cells["tree_shadow_density"] * 0.6).clip(0, 1)

print("Shade potential computed.")
cells[["building_density", "tree_shadow_density", "shade_potential"]].describe().round(3)

In [ ]:
# map of the shade potential
m_shade = cells_to_gdf(cells).explore(
    column='shade_potential',
    cmap='YlGnBu',
    tiles=BASEMAP,
    style_kwds=dict(weight=1, fillOpacity=0.75),
    tooltip=['shade_potential', 'building_density', 'tree_shadow_density', 'ndvi', 'lst'],
    legend_kwds=dict(caption='Estimated shade potential (buildings + trees)')
)

# satellite background
folium.TileLayer(xyz.Esri.WorldImagery, name="Satellite (Esri)", overlay=False).add_to(m_shade)
folium.LayerControl().add_to(m_shade)

m_shade

In [ ]:

cells[['shade_potential', 'building_density', 'tree_shadow_density', 'lst', 'stress_rate']].corr(method='spearman').round(3)

**Shade-optimised routing.** We join `shade_potential` to the street network and add a third cost, `shade_cost = length × (1 − 0.8 × shade_potential)`, which makes shaded streets "shorter". How much longer is the shaded route than the shortest one?

In [ ]:
import osmnx as ox
import networkx as nx
from shapely.geometry import Point, LineString
import geopandas as gpd
import pandas as pd
import folium
import xyzservices.providers as xyz

# Ensure we have our street graph with the assigned environmental variables
G_resilient = ox.graph_to_gdfs(G_weighted, nodes=False, edges=True)

# Regenerate cells_gdf to make sure it includes the newly calculated shade_potential
cells_gdf_updated = cells_to_gdf(cells)

# Join shade_potential from our grid cells back to the street network
# We take the centroids of street edges to match them to our cells
G_resilient['centroid'] = G_resilient.geometry.centroid
gdf_centroids = gpd.GeoDataFrame(G_resilient[['centroid']], geometry='centroid', crs=CRS)
joined_shade = gpd.sjoin(gdf_centroids, cells_gdf_updated[['shade_potential', 'geometry']], how='left', predicate='within')

# Assign back with fallbacks
edges['shade_potential'] = joined_shade['shade_potential'].reindex(edges.index).fillna(cells['shade_potential'].mean())

# Define our custom cost weights:
# 1. Stress Cost: Penalizes segments with high biometric stress
edges['stress_cost'] = edges['length'] * (1.0 + (edges['stress_rate'] / 100.0))

# 2. Shade-Prioritized Cost: Heavily discounts routes with high shading (lowering the 'cost' of shaded paths)
edges['shade_cost'] = edges['length'] * (1.0 - (edges['shade_potential'] * 0.8))

# Rebuild the weighted graph
G_weighted_resilient = ox.graph_from_gdfs(nodes, edges)

# Set Origin and Destination coords (e.g., commute from south-west suburbs to north-east tech park)
orig_node = ox.nearest_nodes(G_weighted_resilient, X=433200, Y=5791200)
dest_node = ox.nearest_nodes(G_weighted_resilient, X=436100, Y=5793000)

# Calculate paths
path_shortest = nx.shortest_path(G_weighted_resilient, orig_node, dest_node, weight='length')
path_calm = nx.shortest_path(G_weighted_resilient, orig_node, dest_node, weight='stress_cost')
path_shaded = nx.shortest_path(G_weighted_resilient, orig_node, dest_node, weight='shade_cost')

# Compute lengths
len_short = sum(G_weighted_resilient[u][v][0]['length'] for u, v in zip(path_shortest[:-1], path_shortest[1:]))
len_calm = sum(G_weighted_resilient[u][v][0]['length'] for u, v in zip(path_calm[:-1], path_calm[1:]))
len_shaded = sum(G_weighted_resilient[u][v][0]['length'] for u, v in zip(path_shaded[:-1], path_shaded[1:]))

# Convert to GeoDataFrames
def route_to_geom(graph, route, color, name, dist):
    nodes_data = graph.nodes(data=True)
    pts = [Point(nodes_data[node]['x'], nodes_data[node]['y']) for node in route]
    geom = LineString(pts)
    return gpd.GeoDataFrame(geometry=[geom], crs=CRS).assign(Route=name, Color=color, Distance=f"{round(dist/1000.0, 2)} km")

gdf_short = route_to_geom(G_weighted_resilient, path_shortest, 'blue', 'Shortest Route', len_short)
gdf_calm_route = route_to_geom(G_weighted_resilient, path_calm, 'red', 'Biometrically Calm Route', len_calm)
gdf_shaded_route = route_to_geom(G_weighted_resilient, path_shaded, 'purple', 'Shaded Canopy Route', len_shaded)

all_routes = pd.concat([gdf_short, gdf_calm_route, gdf_shaded_route])

# Visualize interactive routing options
m_resilient_routing = all_routes.explore(
    column='Route',
    cmap=['blue', 'red', 'purple'],
    tiles=BASEMAP,
    style_kwds=dict(weight=6),
    tooltip=['Route', 'Distance']
)

folium.TileLayer(xyz.Esri.WorldImagery, name="Satellite (Esri)", overlay=False).add_to(m_resilient_routing)
folium.LayerControl().add_to(m_resilient_routing)

print(f"Shortest Route Length: {len_short/1000:.2f} km")
print(f"Biometrically Calm Route Length: {len_calm/1000:.2f} km")
print(f"Maximum Shade Route Length: {len_shaded/1000:.2f} km")
display(m_resilient_routing)

### 📚 References
- Biljecki, F. & Ito, K. (2021). Street view imagery in urban analytics and GIS: A review. *Landscape and Urban Planning* 215, 104217.
- Cordts, M. et al. (2016). The Cityscapes Dataset for Semantic Urban Scene Understanding. *CVPR*.
- Getis, A. & Ord, J. K. (1992). The analysis of spatial association by use of distance statistics. *Geographical Analysis* 24(3), 189–206.
- Haklay, M. (2010). How good is volunteered geographical information? *Environment and Planning B* 37(4), 682–703.
- Jocher, G., Qiu, J. & Chaurasia, A. (2026). *Ultralytics YOLO26.* https://docs.ultralytics.com/models/yolo26
- Kyriakou, K., Resch, B. et al. (2019). Detecting Moments of Stress from Measurements of Wearable Physiological Sensors. *Sensors* 19(17), 3805.
- Li, X. et al. (2015). Assessing street-level urban greenery using Google Street View and a modified green view index. *Urban Forestry & Urban Greening* 14(3), 675–685.
- Lin, T.-Y. et al. (2014). Microsoft COCO: Common Objects in Context. *ECCV*.
- Openshaw, S. (1984). *The Modifiable Areal Unit Problem.* CATMOG 38.
- Redmon, J. et al. (2016). You Only Look Once: Unified, Real-Time Object Detection. *CVPR*.
- Tobler, W. (1970). A computer movie simulating urban growth in the Detroit region. *Economic Geography* 46, 234–240.
- Tucker, C. J. (1979). Red and photographic infrared linear combinations for monitoring vegetation. *Remote Sensing of Environment* 8, 127–150.
- Voogt, J. A. & Oke, T. R. (2003). Thermal remote sensing of urban climates. *Remote Sensing of Environment* 86, 370–384.
- Xie, E. et al. (2021). SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers. *NeurIPS*.
- Zha, Y., Gao, J. & Ni, S. (2003). Use of normalized difference built-up index in automatically mapping urban areas from TM imagery. *International Journal of Remote Sensing* 24(3), 583–594.

*Data: Sentinel-2 © Copernicus/ESA; Landsat courtesy of USGS; © OpenStreetMap contributors (ODbL); street view images © Mapillary contributors (CC-BY-SA).*